
# Árboles y ensambles, Notebook 1
## Entropía e información: medir cuánto sabemos y cuánto aprendemos

**Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De qué se trata

Antes de construir un árbol de decisión hay que responder una pregunta que parece filosófica
y resulta muy práctica: **¿cómo se mide cuánto sabemos?** Si un banco tiene 14 clientes y
8 pagaron, ¿cuánta incertidumbre hay sobre el próximo? Y si además sabemos que el cliente tiene
poca deuda, ¿cuánto de esa incertidumbre desaparece?

Claude Shannon respondió eso en 1948 con un número: la **entropía**. Y la reducción de
entropía al conocer un dato, la **información mutua** (o ganancia de información), resulta ser
exactamente lo que un árbol de decisión usa para elegir qué pregunta hacer primero.

Este notebook usa una tabla de 14 empresas para que cada cálculo se pueda comprobar con los
dedos, y después simulaciones para ver lo que no se ve con 14 filas: qué pasa cuando el
experimento se repite muchas veces.

### Qué vas a aprender hoy

1. Qué es la información de un evento y por qué se mide con un logaritmo.
2. Qué es la entropía de una variable y cómo se calcula con conteos.
3. Qué es la entropía condicional y la ganancia de información (información mutua).
4. Cómo se relacionan con lo que ya conoces: la entropía es un pariente de la **varianza**
   y la información mutua, un pariente de la **correlación**, con una diferencia importante:
   la correlación solo ve relaciones en línea recta; la información mutua ve cualquiera.


In [17]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("Listo.")


Listo.



## 1. La información de un evento: la sorpresa

Un evento seguro no informa nada. Uno raro informa mucho. Shannon puso número a esa intuición:

$$
I(\text{evento}) = -\log_2(p)
$$

donde $p$ es la probabilidad del evento y el resultado se mide en **bits**. Una moneda justa
($p = 0{,}5$) informa 1 bit; un dado ($p = 1/6$) informa 2,58 bits; un evento seguro ($p = 1$),
0 bits.

**Por qué el logaritmo.** Para que dos eventos independientes *sumen* su información:
$-\log_2(p \cdot q) = -\log_2(p) - \log_2(q)$. Dos monedas informan 2 bits, no "0,25 de algo".
El logaritmo es la única función que tiene esa propiedad.


In [18]:

eventos = pd.DataFrame({
    "evento": ["sale cara (moneda justa)", "sale un 3 (dado)", "sale el 17 (ruleta de 36)",
               "una empresa AAA cae en impago este año", "una empresa con rating C cae en impago este año", "el sol sale mañana"],
    "p": [0.5, 1/6, 1/36, 0.01, 0.6, 1.0],
})
eventos["informacion_bits"] = -np.log2(eventos["p"])
eventos.round(3)


,evento,p,informacion_bits
0,sale cara (moneda justa),0.500,1.000
1,sale un 3 (dado),0.167,2.585
2,sale el 17 (ruleta de 36),0.028,5.170
3,una empresa AAA cae en impago este año,0.010,6.644
4,una empresa con rating C cae en impago este año,0.600,0.737
5,el sol sale mañana,1.000,-0.000


In [19]:

p = np.linspace(0.01, 1, 200)
fig = px.line(x=p, y=-np.log2(p), labels={"x": "probabilidad p", "y": "información, en bits"},
              title="Información de un evento según su probabilidad: −log₂(p)")
fig.show()



## 2. La entropía: la sorpresa promedio

Si cada valor de una variable informa $-\log_2 p(v)$, la variable completa informa, en promedio:

$$
H(X) = -\sum_{v} p(v) \, \log_2 p(v)
$$

Lo que significa cada cosa: $v$ recorre los valores posibles de $X$ y $p(v)$ es la fracción de
veces que sale cada uno. $H$ se lee como "cuántas preguntas de sí o no hacen falta, en
promedio, para adivinar el valor". Una moneda justa: 1 bit. Una moneda cargada al 90%: 0,47
bits (casi siempre sabemos qué va a salir). Una constante: 0.

Veámoslo en las 14 empresas del curso.


In [20]:

# Las 14 empresas que pidieron crédito (la tabla chica del curso)
credito = pd.DataFrame({
    "tamano":    ["pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande"],
    "deuda":     ["baja", "baja", "baja", "baja", "alta", "alta", "alta", "alta", "alta", "media", "media", "media", "media", "media"],
    "garantia":  ["no", "si", "no", "si", "si", "si", "no", "no", "no", "no", "si", "si", "no", "no"],
    "historial": ["bueno", "malo", "malo", "bueno", "bueno", "malo", "bueno", "malo", "bueno", "bueno", "bueno", "malo", "malo", "malo"],
    "paga":      ["si", "si", "si", "si", "si", "si", "no", "no", "no", "si", "si", "no", "no", "no"],
})
credito.index = range(1, 15)
credito


,tamano,deuda,garantia,historial,paga
1,pequena,baja,no,bueno,si
2,grande,baja,si,malo,si
3,mediana,baja,no,malo,si
4,pequena,baja,si,bueno,si
5,grande,alta,si,bueno,si
6,mediana,alta,si,malo,si
7,pequena,alta,no,bueno,no
8,grande,alta,no,malo,no
9,mediana,alta,no,bueno,no
10,pequena,media,no,bueno,si


In [21]:

def entropia(serie):
    # Entropía en bits de una columna: contar, pasar a probabilidad, sumar -p log2 p
    p = serie.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

for columna in credito.columns:
    print(f"H({columna:9s}) = {entropia(credito[columna]):.3f} bits    valores: {dict(credito[columna].value_counts())}")


H(tamano   ) = 1.577 bits    valores: {'pequena': np.int64(5), 'grande': np.int64(5), 'mediana': np.int64(4)}
H(deuda    ) = 1.577 bits    valores: {'alta': np.int64(5), 'media': np.int64(5), 'baja': np.int64(4)}
H(garantia ) = 0.985 bits    valores: {'no': np.int64(8), 'si': np.int64(6)}
H(historial) = 1.000 bits    valores: {'bueno': np.int64(7), 'malo': np.int64(7)}
H(paga     ) = 0.985 bits    valores: {'si': np.int64(8), 'no': np.int64(6)}



**Cómo leerlo.** 8 de 14 empresas pagan, así que $p(\text{si}) = 0{,}571$ y
$H(\text{paga}) = 0{,}985$ bits: casi el máximo de 1 bit. Adivinar si una empresa paga es casi
una moneda justa. Las variables con tres valores (tamaño, deuda) pueden llegar hasta
$\log_2 3 = 1{,}585$ bits.

La curva completa para una variable de dos valores: la entropía es máxima cuando las dos
opciones son igual de probables y cae a cero en los extremos.


In [22]:

p = np.linspace(0.001, 0.999, 300)
H = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
fig = px.line(x=p, y=H, labels={"x": "p(si)", "y": "H, en bits"}, title="Entropía de una variable con dos valores")
p_paga = (credito["paga"] == "si").mean()
fig.add_trace(go.Scatter(x=[p_paga], y=[entropia(credito["paga"])], mode="markers+text", text=["paga"], textposition="top center",
                         marker=dict(size=12, color="red"), name="paga en las 14 empresas"))
fig.show()



## 3. Entropía condicional y ganancia: cuánto aprendo al preguntar

Si conozco el nivel de deuda de una empresa, ¿cuánta incertidumbre queda sobre si paga? Se
calcula la entropía de "paga" **dentro de cada grupo** de deuda y se promedia, pesando cada
grupo por su tamaño:

$$
H(Y \mid X) = \sum_{v} P(X = v) \cdot H(Y \text{ dentro del grupo } X = v)
$$

Y la **ganancia de información**, también llamada **información mutua**, es lo que desaparece:

$$
I(Y; X) = H(Y) - H(Y \mid X)
$$

Aquí $Y$ es la variable que queremos predecir (paga) y $X$ el atributo que estamos considerando
preguntar. Un atributo con ganancia alta es una buena pregunta; con ganancia cero, una pregunta
inútil.


In [23]:

def entropia_condicional(df, objetivo, atributo):
    # Promedio ponderado de la entropía del objetivo dentro de cada grupo del atributo
    total = 0.0
    for valor, grupo in df.groupby(atributo):
        peso = len(grupo) / len(df)
        total += peso * entropia(grupo[objetivo])
    return total

def ganancia(df, objetivo, atributo):
    return entropia(df[objetivo]) - entropia_condicional(df, objetivo, atributo)

# El detalle para la deuda, grupo por grupo
print("H(paga) =", round(entropia(credito["paga"]), 3))
for valor, grupo in credito.groupby("deuda"):
    print(f"  deuda = {valor:6s}: {len(grupo)} empresas, {int((grupo.paga == 'si').sum())} pagan, "
          f"H = {entropia(grupo.paga):.3f}, peso {len(grupo)/14:.3f}, aporta {len(grupo)/14 * entropia(grupo.paga):.3f}")
print("H(paga | deuda) =", round(entropia_condicional(credito, "paga", "deuda"), 3))
print("ganancia(deuda) =", round(ganancia(credito, "paga", "deuda"), 3), "bits")


H(paga) = 0.985
  deuda = alta  : 5 empresas, 2 pagan, H = 0.971, peso 0.357, aporta 0.347
  deuda = baja  : 4 empresas, 4 pagan, H = -0.000, peso 0.286, aporta -0.000
  deuda = media : 5 empresas, 2 pagan, H = 0.971, peso 0.357, aporta 0.347
H(paga | deuda) = 0.694
ganancia(deuda) = 0.292 bits


In [24]:

tabla = pd.DataFrame({
    "atributo": ["tamano", "deuda", "garantia", "historial"],
})
tabla["H(paga)"] = entropia(credito["paga"])
tabla["H(paga | X)"] = [entropia_condicional(credito, "paga", a) for a in tabla["atributo"]]
tabla["ganancia"] = tabla["H(paga)"] - tabla["H(paga | X)"]
tabla["H(X)"] = [entropia(credito[a]) for a in tabla["atributo"]]
tabla["gain ratio"] = tabla["ganancia"] / tabla["H(X)"]
tabla.round(3)


,atributo,H(paga),H(paga | X),ganancia,H(X),gain ratio
0,tamano,0.985,0.979,0.006,1.577,0.004
1,deuda,0.985,0.694,0.292,1.577,0.185
2,garantia,0.985,0.824,0.161,0.985,0.164
3,historial,0.985,0.924,0.061,1.000,0.061



**Qué dice la tabla.** La deuda gana con 0,292 bits: preguntar por la deuda deja casi un
tercio menos de incertidumbre. El tamaño casi no aporta (0,006). Esa tabla es, literalmente,
la primera decisión de un árbol: la raíz es el atributo de mayor ganancia. Lo vamos a construir
en el Notebook 2.

**El gain ratio y la trampa del identificador.** Un atributo con muchos valores parte la tabla
en pedacitos y gana "gratis". El caso extremo es un identificador: un valor por empresa,
$H(\text{paga} \mid \text{id}) = 0$, ganancia máxima e inútil (una empresa nueva trae un id que
nunca vimos). C4.5 divide la ganancia por la entropía del atributo, $H(X)$, para castigar eso.


In [25]:

con_id = credito.copy()
con_id["id"] = range(1, 15)
g = ganancia(con_id, "paga", "id"); h = entropia(con_id["id"])
print(f"ganancia(id) = {g:.3f} bits (la máxima posible)   H(id) = {h:.3f}   gain ratio = {g/h:.3f}")
print("El gain ratio del id (0,259) todavía le gana al de la deuda (0,185) en una tabla de 14 filas.")
print("Por eso C4.5 exige además que la ganancia supere el promedio, y por eso el sentido común manda:")
print("un atributo que identifica filas no puede generalizar.")


ganancia(id) = 0.985 bits (la máxima posible)   H(id) = 3.807   gain ratio = 0.259
El gain ratio del id (0,259) todavía le gana al de la deuda (0,185) en una tabla de 14 filas.
Por eso C4.5 exige además que la ganancia supere el promedio, y por eso el sentido común manda:
un atributo que identifica filas no puede generalizar.



## 4. El paralelo con la estadística que ya conoces

### Entropía y varianza

La varianza mide cuánto se dispersa un **número** alrededor de su promedio. La entropía mide
cuánta incertidumbre tiene una variable **de cualquier tipo**, y no tiene unidades. Las dos
son medidas de dispersión, y las dos valen cero cuando la variable es constante. Pero:

- la entropía solo mira las **probabilidades**: tres clases iguales valen $\log_2 3$ bits, sean
  1, 2, 3 o 10, 20, 30;
- la varianza mira los **valores** y sus unidades: pasar de 1, 2, 3 a 10, 20, 30 la multiplica
  por 100.

Por eso la entropía sirve para variables sin números (paga sí/no, tamaño chico/mediano/grande),
donde la varianza no tiene sentido. Y por eso un árbol de **clasificación** usa entropía o Gini
exactamente donde un árbol de **regresión** usa varianza: como medida de impureza que un buen
corte reduce.


In [26]:

def entropia_de_probabilidades(p):
    p = np.array(p); p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def varianza_discreta(valores, p):
    valores = np.array(valores, dtype=float); p = np.array(p)
    media = (valores * p).sum()
    return float((p * (valores - media) ** 2).sum())

distribuciones = [
    ("constante (5)",                    [5],                [1.0]),
    ("moneda justa (0 o 1)",             [0, 1],             [0.5, 0.5]),
    ("moneda cargada (0,9 / 0,1)",       [0, 1],             [0.9, 0.1]),
    ("dado justo (1 a 6)",               [1, 2, 3, 4, 5, 6], [1/6] * 6),
    ("tres clases iguales (1, 2, 3)",    [1, 2, 3],          [1/3] * 3),
    ("tres clases iguales (10, 20, 30)", [10, 20, 30],       [1/3] * 3),
]
pd.DataFrame([(n, len(v), entropia_de_probabilidades(p), varianza_discreta(v, p)) for n, v, p in distribuciones],
             columns=["distribución", "estados", "entropía (bits)", "varianza"]).round(3)


,distribución,estados,entropía (bits),varianza
0,constante (5),1,-0.000,0.000
1,moneda justa (0 o 1),2,1.000,0.250
2,"moneda cargada (0,9 / 0,1)",2,0.469,0.090
3,dado justo (1 a 6),6,2.585,2.917
4,"tres clases iguales (1, 2, 3)",3,1.585,0.667
5,"tres clases iguales (10, 20, 30)",3,1.585,66.667



Para una variable **continua** la entropía necesita agrupar los valores en clases (como un
histograma), y el resultado depende de cuántas clases se usen. La varianza no. Un
experimento: cuatro variables con 2.000 valores cada una, agrupadas en 10 clases de igual ancho.


In [27]:

def entropia_continua(x, clases=10):
    # Agrupa en clases de igual ancho entre el mínimo y el máximo y calcula H con las frecuencias
    conteos, _ = np.histogram(x, bins=clases)
    return entropia_de_probabilidades(conteos / conteos.sum())

rng = np.random.default_rng(7)
n = 2000
muestras = {
    "normal(0, 1)":        rng.normal(0, 1, n),
    "normal(0, 3)":        rng.normal(0, 3, n),
    "uniforme(-1,7, 1,7)": rng.uniform(-1.7, 1.7, n),   # misma varianza que normal(0, 1)
    "lognormal(0, 1)":     rng.lognormal(0, 1, n),
}
pd.DataFrame({nombre: {"varianza": np.var(x), "desviación estándar": np.std(x),
                       "entropía (10 clases)": entropia_continua(x, 10), "entropía (30 clases)": entropia_continua(x, 30)}
              for nombre, x in muestras.items()}).T.round(3)


,varianza,desviación estándar,entropía (10 clases),entropía (30 clases)
"normal(0, 1)",0.970,0.985,2.726,4.281
"normal(0, 3)",8.959,2.993,2.609,4.167
"uniforme(-1,7, 1,7)",0.988,0.994,3.319,4.898
"lognormal(0, 1)",3.943,1.986,0.372,1.494



**Qué mirar.** normal(0, 1) y normal(0, 3) tienen casi la misma entropía (la forma es la misma,
solo cambia la escala) y varianzas 9 veces distintas. La uniforme tiene la misma varianza que
normal(0, 1) y más entropía: reparte sus valores más parejo entre las clases. La lognormal tiene
varianza grande por su cola larga y poca entropía: casi todo cae en las clases bajas. La
entropía mide *qué tan repartida* está la variable; la varianza, *qué tan lejos* llega. Y con
30 clases en vez de 10 todas las entropías suben: para variables continuas el número depende
de cómo se agrupa.

### Información mutua y correlación

La correlación de Pearson mide cuánto se parece la relación entre $X$ e $Y$ a una **recta**
(de $-1$ a $1$). La información mutua mide cuánto reduce $X$ la incertidumbre de $Y$, **sea
cual sea la forma** de la relación (en bits, de 0 hacia arriba). Con las variables agrupadas
en clases:

$$
I(X; Y) = \sum_{x, y} p(x, y) \, \log_2 \frac{p(x, y)}{p(x) \, p(y)}
$$

donde $p(x, y)$ es la fracción de observaciones en la celda (clase de $X$, clase de $Y$) y
$p(x)$, $p(y)$ las fracciones de cada clase por separado. Si $X$ e $Y$ fueran independientes,
$p(x, y) = p(x) \, p(y)$ en todas las celdas y la suma daría cero.

El experimento decisivo: tres relaciones simuladas, una recta, una **U** y puro azar.


In [28]:

def informacion_mutua(x, y, clases=3):
    # Agrupa x e y en clases de igual ancho y calcula I(X; Y) con la tabla conjunta
    cx = pd.cut(x, bins=clases, labels=False)
    cy = pd.cut(y, bins=clases, labels=False)
    conjunta = pd.crosstab(cx, cy, normalize=True).values
    px = conjunta.sum(axis=1, keepdims=True); py = conjunta.sum(axis=0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        aportes = np.where(conjunta > 0, conjunta * np.log2(conjunta / (px * py)), 0.0)
    return float(aportes.sum())

rng = np.random.default_rng(3)
n = 300
x = rng.uniform(0, 1, n)
ruido = rng.normal(0, 0.05, n)
simulado = pd.DataFrame({
    "x": x,
    "y_recta": 0.3 + 0.55 * x + ruido,
    "y_U": 0.1 + 0.9 * (x - 0.5) ** 2 + ruido,
    "y_azar": 0.2 + 0.6 * rng.uniform(0, 1, n) + ruido,
})
fig = px.scatter(simulado.melt(id_vars="x", var_name="relación", value_name="y"), x="x", y="y", facet_col="relación", opacity=0.6,
                 title="Tres relaciones: una recta, una U y azar")
fig.show()

resumen = pd.DataFrame({
    "correlación": [simulado["x"].corr(simulado[c]) for c in ["y_recta", "y_U", "y_azar"]],
    "información mutua (bits, 3 clases)": [informacion_mutua(simulado["x"], simulado[c], 3) for c in ["y_recta", "y_U", "y_azar"]],
    "información mutua (bits, 5 clases)": [informacion_mutua(simulado["x"], simulado[c], 5) for c in ["y_recta", "y_U", "y_azar"]],
}, index=["recta", "U", "azar"])
resumen.round(3)


,correlación,"información mutua (bits, 3 clases)","información mutua (bits, 5 clases)"
recta,0.947,0.840,1.209
U,-0.042,0.154,0.436
azar,0.016,0.003,0.033



**Qué dice la tabla.** La recta: las dos medidas la ven. La U: la correlación queda cerca de
cero porque la mitad que sube cancela a la mitad que baja, pero la información mutua es
claramente mayor que la del azar. Saber $x$ **sí** dice mucho sobre $y$; lo que no hay es una
recta. El azar: correlación e información mutua cerca de cero (la información mutua nunca es
exactamente cero con datos finitos: los conteos tienen ruido, y ese es el piso contra el que
hay que comparar).

**Por qué importa para lo que viene.** Un árbol elige sus cortes por ganancia de información
(o por Gini, o por varianza), sin suponer ninguna forma. Con dos cortes en $x$ encuentra la U.
Una regresión lineal sobre $x$ la ignora por completo. Probémoslo.


In [29]:

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

X_ = simulado[["x"]]
for objetivo in ["y_recta", "y_U"]:
    lineal = LinearRegression().fit(X_, simulado[objetivo])
    arbol = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_, simulado[objetivo])
    print(f"{objetivo:8s}  R² regresión lineal = {lineal.score(X_, simulado[objetivo]):.3f}   R² árbol de 2 niveles = {arbol.score(X_, simulado[objetivo]):.3f}")


y_recta   R² regresión lineal = 0.897   R² árbol de 2 niveles = 0.856
y_U       R² regresión lineal = 0.002   R² árbol de 2 niveles = 0.515



El árbol de dos niveles (cuatro escalones) explica la U casi tan bien como la recta; la
regresión lineal explica la U prácticamente en nada. Esa es la razón de fondo por la que los
árboles y sus ensambles dominan los datos tabulares: no necesitan que nadie les diga la forma.

## 5. Lo que hay que llevarse

- La **información** de un evento es $-\log_2 p$; la **entropía** es la información promedio
  de una variable, y mide cuánto nos falta saber.
- La **ganancia de información** (información mutua) de un atributo es cuánta entropía del
  objetivo desaparece al conocerlo. Es la regla con la que un árbol elige sus preguntas.
- Entropía y varianza son dos medidas de dispersión: la varianza necesita números y escala; la
  entropía, solo probabilidades.
- Correlación e información mutua son dos medidas de relación: la correlación solo ve rectas;
  la información mutua ve cualquier forma. Una correlación baja no siempre significa "no hay
  relación".

## Ejercicios

**Ejercicio 1.** Calcula la ganancia de información de cada atributo, pero solo con las 5
empresas de deuda alta (`credito[credito.deuda == "alta"]`). ¿Qué atributo gana ahora? Es la
segunda pregunta que hará el árbol por esa rama.

**Ejercicio 2.** En el experimento de la U, cambia el ruido de 0,05 a 0,20 y repite la tabla de
correlación e información mutua. ¿Cuánto se acerca la U al azar? ¿Y con 3.000 puntos en vez de
300?

**Ejercicio 3.** Toma la tabla de 219 empresas del curso (está en el Notebook 2, sección 4) y
calcula, para cada uno de los cinco ratios, su correlación con `impago` y su información mutua
con `impago` (con 5 clases). ¿Ordenan igual a los ratios?



## Soluciones


In [30]:

# SOLUCIÓN 1 -- la rama de deuda alta
alta = credito[credito.deuda == "alta"]
for a in ["tamano", "garantia", "historial"]:
    print(f"ganancia({a:9s}) en deuda alta = {ganancia(alta, 'paga', a):.3f} bits")
print("Gana la garantía: dentro de las empresas con deuda alta, tener garantía separa perfectamente a las que pagan.")


ganancia(tamano   ) en deuda alta = 0.171 bits
ganancia(garantia ) en deuda alta = 0.971 bits
ganancia(historial) en deuda alta = 0.020 bits
Gana la garantía: dentro de las empresas con deuda alta, tener garantía separa perfectamente a las que pagan.


In [31]:

# SOLUCIÓN 2 -- más ruido, más datos
for n_, sd in [(300, 0.05), (300, 0.20), (3000, 0.05), (3000, 0.20)]:
    r = np.random.default_rng(3)
    x_ = r.uniform(0, 1, n_); e_ = r.normal(0, sd, n_)
    yu = 0.1 + 0.9 * (x_ - 0.5) ** 2 + e_; ya = 0.2 + 0.6 * r.uniform(0, 1, n_) + e_
    print(f"n = {n_:5d}, ruido = {sd:.2f}:  U -> corr {np.corrcoef(x_, yu)[0,1]:6.3f}, IM {informacion_mutua(x_, yu):.3f} bits;"
          f"   azar -> corr {np.corrcoef(x_, ya)[0,1]:6.3f}, IM {informacion_mutua(x_, ya):.3f} bits")
print("Con más ruido la U se acerca al azar; con más datos el piso del azar baja y la U se distingue mejor.")


n =   300, ruido = 0.05:  U -> corr -0.042, IM 0.154 bits;   azar -> corr  0.016, IM 0.003 bits
n =   300, ruido = 0.20:  U -> corr -0.067, IM 0.011 bits;   azar -> corr -0.028, IM 0.008 bits
n =  3000, ruido = 0.05:  U -> corr -0.035, IM 0.175 bits;   azar -> corr -0.009, IM 0.000 bits
n =  3000, ruido = 0.20:  U -> corr -0.017, IM 0.025 bits;   azar -> corr -0.008, IM 0.001 bits
Con más ruido la U se acerca al azar; con más datos el piso del azar baja y la U se distingue mejor.


In [32]:

# SOLUCIÓN 3 -- los ratios de la tabla de impago
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
sol = pd.DataFrame({
    "correlación con impago": [impago[r].corr(impago["impago"]) for r in ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]],
    "información mutua (5 clases)": [informacion_mutua(impago[r], impago["impago"].astype(float), 5) for r in ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]],
}, index=["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"])
print(sol.round(3))
print()
print("Nota: 'impago' solo tiene dos valores, así que sus 5 clases se reducen a 2. Compara el orden por |correlación| con el orden por información mutua.")


                 correlación con impago  información mutua (5 clases)
deuda_activos                     0.433                         0.114
razon_corriente                  -0.215                         0.019
ventas_deuda                     -0.214                         0.020
ln_activos                       -0.188                         0.063
roa                              -0.103                         0.029

Nota: 'impago' solo tiene dos valores, así que sus 5 clases se reducen a 2. Compara el orden por |correlación| con el orden por información mutua.
